# 📓 Day 2 Assignment: Reference Solution

This notebook provides a complete, production-grade reference solution for the Day 2 assignment. It implements the **Smart Contact Book** and **Personal Expense Tracker** as a unified command-line interface, including the **Bonus Challenges**:
1. **JSON File Persistence** (saving/loading data dynamically, including converting sets to lists for serialization).
2. **Category Tag Venn Analysis** (set intersections to query contacts matching multiple labels).

## 1. JSON Persistence Layer
Since JSON does not natively support Python sets, we must write utility functions to convert tags from lists (in JSON) to sets (in Python memory) and vice-versa.

In [ ]:
import json
import os

CONTACTS_FILE = 'contacts.json'
EXPENSES_FILE = 'expenses.json'

def load_contacts() -> dict:
    """
    Loads contacts from a JSON file and converts tags list back to Python sets.
    """
    if not os.path.exists(CONTACTS_FILE):
        return {}
    try:
        with open(CONTACTS_FILE, 'r') as f:
            data = json.load(f)
            # Convert tags list to set for each contact
            for phone, details in data.items():
                details['tags'] = set(details.get('tags', []))
            return data
    except Exception as e:
        print(f'Error loading contacts: {e}')
        return {}

def save_contacts(contacts: dict):
    """
    Converts contact tags sets to lists and serializes to JSON file.
    """
    try:
        # Create a copy to prevent mutating in-memory sets
        serializable_contacts = {}
        for phone, details in contacts.items():
            serializable_contacts[phone] = {
                'name': details['name'],
                'email': details['email'],
                'tags': list(details['tags'])
            }
        with open(CONTACTS_FILE, 'w') as f:
            json.dump(serializable_contacts, f, indent=4)
    except Exception as e:
        print(f'Error saving contacts: {e}')

def load_expenses() -> list:
    """
    Loads expenses from JSON file.
    """
    if not os.path.exists(EXPENSES_FILE):
        return []
    try:
        with open(EXPENSES_FILE, 'r') as f:
            return json.load(f)
    except Exception as e:
        print(f'Error loading expenses: {e}')
        return []

def save_expenses(expenses: list):
    """
    Saves expenses list to JSON file.
    """
    try:
        with open(EXPENSES_FILE, 'w') as f:
            json.dump(expenses, f, indent=4)
    except Exception as e:
        print(f'Error saving expenses: {e}')

## 2. Module 1: Smart Contact Book Operations
Here we implement core contacts operations using dictionaries, lookups, and sets.

In [ ]:
def add_contact(contacts: dict, name: str, phone: str, email: str, tags: list) -> bool:
    """
    Adds a contact. Returns True if added/updated, False otherwise.
    """
    phone = phone.strip()
    if not phone.isdigit() or len(phone) < 10:
        print('Error: Invalid phone number. Must be at least 10 digits.')
        return False
        
    if phone in contacts:
        existing_name = contacts[phone]['name']
        print(f'Warning: Contact with phone {phone} already exists: {existing_name}.')
        overwrite = input('Do you want to overwrite this contact? (y/n): ').lower().strip()
        if overwrite != 'y':
            print('Operation cancelled.')
            return False
            
    # Convert tags input list to a unique Set
    tag_set = {t.strip().title() for t in tags if t.strip()}
    
    contacts[phone] = {
        'name': name.strip().title(),
        'email': email.strip().lower(),
        'tags': tag_set
    }
    saved_name = contacts[phone]['name']
    print(f'Contact {saved_name} saved successfully!')
    return True

def search_contact(contacts: dict, query: str):
    """
    Searches for contacts by phone (O(1)) or matches name (O(N)).
    """
    query = query.strip()
    results = {}
    
    # Direct lookup (O(1))
    if query in contacts:
        results[query] = contacts[query]
    else:
        # Partial name search (O(N))
        query_lower = query.lower()
        for phone, info in contacts.items():
            if query_lower in info['name'].lower():
                results[phone] = info
                
    return results

def delete_contact(contacts: dict, phone: str) -> bool:
    """
    Deletes contact by phone number.
    """
    phone = phone.strip()
    if phone in contacts:
        removed_name = contacts.pop(phone)['name']
        print(f'Contact {removed_name} deleted successfully!')
        return True
    print('Error: Contact not found.')
    return False

def intersect_tags(contacts: dict, tag1: str, tag2: str):
    """
    Bonus: Finds contacts matching BOTH tags using set intersection.
    """
    t1 = tag1.strip().title()
    t2 = tag2.strip().title()
    matched = {}
    
    for phone, info in contacts.items():
        query_tags = {t1, t2}
        if query_tags.issubset(info['tags']):
            matched[phone] = info
            
    return matched

## 3. Module 2: Personal Expense Tracker Operations
Here we implement expense operations, validations, and analytics using list comprehensions.

In [ ]:
ALLOWED_CATEGORIES = {'food', 'transport', 'supplies', 'others'}

def add_expense(expenses: list, desc: str, amount: float, category: str) -> bool:
    """
    Validates category, constructs dictionary, and appends to expenses list.
    """
    cat_lower = category.strip().lower()
    if cat_lower not in ALLOWED_CATEGORIES:
        print(f'Error: Invalid category. Must be one of: {ALLOWED_CATEGORIES}')
        return False
    if amount <= 0:
        print('Error: Amount must be greater than 0.')
        return False
        
    expenses.append({
        'description': desc.strip(),
        'amount': float(amount),
        'category': cat_lower
    })
    print(f'Expense logged: {desc} - Rs. {amount:.2f}')
    return True

def filter_expenses_by_category(expenses: list, category: str) -> list:
    """
    Use list comprehension to filter items belonging to the target category.
    """
    cat_lower = category.strip().lower()
    return [e for e in expenses if e['category'] == cat_lower]

def filter_high_value_expenses(expenses: list, threshold: float = 500.0) -> list:
    """
    Use list comprehension to filter items costing more than a threshold amount.
    """
    return [e for e in expenses if e['amount'] > threshold]

def calculate_total_expenses(expenses: list) -> float:
    """
    Calculates the sum total of all logged expenses.
    """
    return sum([e['amount'] for e in expenses])

## 4. Unified Interactive CLI Terminal Menu System
The code block below sets up the menu flow and loop handlers. Running this block will launch the interactive console program directly in the notebook environment.

In [ ]:
def print_contact_card(phone, info):
    name_val = info.get('name')
    email_val = info.get('email')
    tags_val = ', '.join(info.get('tags', []))
    print(f'Name   : {name_val}')
    print(f'Phone  : {phone}')
    print(f'Email  : {email_val}')
    print(f'Tags   : {tags_val}')
    print('-' * 30)

def manage_contacts_loop(contacts):
    while True:
        print('\n--- CONTACT BOOK SUBMENU ---')
        print('1. Add Contact')
        print('2. View All Contacts')
        print('3. Search Contacts')
        print('4. Delete Contact')
        print('5. Venn Tag Analysis (Intersection)')
        print('6. Back to Main Menu')
        choice = input('Enter option (1-6): ').strip()
        
        if choice == '1':
            name = input('Enter name: ')
            phone = input('Enter phone: ')
            email = input('Enter email: ')
            tags_raw = input('Enter tags (comma-separated, e.g. VIP, Credit): ')
            tags_list = tags_raw.split(',')
            if add_contact(contacts, name, phone, email, tags_list):
                save_contacts(contacts)
                
        elif choice == '2':
            if not contacts:
                print('No contacts found.')
            else:
                print('\n--- ALL CONTACTS ---')
                for phone, info in contacts.items():
                    print_contact_card(phone, info)
                    
        elif choice == '3':
            q = input('Enter Name or Phone Number to search: ')
            results = search_contact(contacts, q)
            if not results:
                print('No matching contacts found.')
            else:
                print(f'Found {len(results)} matches:')
                for phone, info in results.items():
                    print_contact_card(phone, info)
                    
        elif choice == '4':
            phone = input('Enter Phone Number to delete: ')
            if delete_contact(contacts, phone):
                save_contacts(contacts)
                
        elif choice == '5':
            print('Query contacts containing BOTH tags.')
            t1 = input('Enter Tag 1: ')
            t2 = input('Enter Tag 2: ')
            results = intersect_tags(contacts, t1, t2)
            if not results:
                print(f'No contacts match both tags: {t1} and {t2}')
            else:
                print(f'Found {len(results)} matches:')
                for phone, info in results.items():
                    print_contact_card(phone, info)
                    
        elif choice == '6':
            break
        else:
            print('Invalid option. Try again.')

def manage_expenses_loop(expenses):
    while True:
        print('\n--- EXPENSE TRACKER SUBMENU ---')
        print('1. Log Expense')
        print('2. View All Expenses')
        print('3. Filter Expenses by Category')
        print('4. Filter High-Value Expenses (> Rs. 500)')
        print('5. Show Total Outflow Summary')
        print('6. Back to Main Menu')
        choice = input('Enter option (1-6): ').strip()
        
        if choice == '1':
            desc = input('Enter description: ')
            try:
                amount = float(input('Enter amount (Rs.): '))
            except ValueError:
                print('Error: Amount must be a number.')
                continue
            print(f'Allowed Categories: {ALLOWED_CATEGORIES}')
            cat = input('Enter category: ')
            if add_expense(expenses, desc, amount, cat):
                save_expenses(expenses)
                
        elif choice == '2':
            if not expenses:
                print('No expenses logged today.')
            else:
                print('\n--- LOGGED EXPENSES ---')
                for i, e in enumerate(expenses, 1):
                    desc_val = e.get('description', '')
                    amt_val = e.get('amount', 0.0)
                    cat_val = e.get('category', '').title()
                    print(f'{i}. {desc_val:18} | Rs. {amt_val:7.2f} | Category: {cat_val}')

        elif choice == '3':
            cat = input('Enter category to filter: ')
            filtered = filter_expenses_by_category(expenses, cat)
            if not filtered:
                print(f'No expenses found in category {cat}.')
            else:
                print(f'\n--- EXPENSES IN {cat.upper()} ---')
                for e in filtered:
                    desc_val = e.get('description', '')
                    amt_val = e.get('amount', 0.0)
                    print(f'- {desc_val:18} | Rs. {amt_val:.2f}')

        elif choice == '4':
            threshold = 500.0
            filtered = filter_high_value_expenses(expenses, threshold)
            if not filtered:
                print(f'No expenses found above Rs. {threshold:.2f}')
            else:
                print(f'\n--- HIGH-VALUE EXPENSES (> Rs. {threshold:.2f}) ---')
                for e in filtered:
                    desc_val = e.get('description', '')
                    amt_val = e.get('amount', 0.0)
                    cat_val = e.get('category', '').title()
                    print(f'- {desc_val:18} | Rs. {amt_val:.2f} | [{cat_val}]')

        elif choice == '5':
            total = calculate_total_expenses(expenses)
            print('\n📊 Financial Outflow Summary:')
            print(f'- Total cash spent: Rs. {total:.2f}')
            print(f'- Number of logs: {len(expenses)}')
            
        elif choice == '6':
            break
        else:
            print('Invalid option. Try again.')

def main_console_app():
    # Initialize systems and load saved persistence files
    contacts = load_contacts()
    expenses = load_expenses()
    
    while True:
        print('\n=========================================')
        print('       COIMBATORE RETAILERS TOOLKIT      ')
        print('=========================================')
        print('1. Manage Contacts (Contact Book)')
        print('2. Manage Expenses (Expense Tracker)')
        print('3. Exit System')
        print('=========================================')
        choice = input('Enter menu option (1-3): ').strip()
        
        if choice == '1':
            manage_contacts_loop(contacts)
        elif choice == '2':
            manage_expenses_loop(expenses)
        elif choice == '3':
            print('\nThank you for using Coimbatore Retailers Toolkit. Exiting...')
            break
        else:
            print('Invalid main menu option. Please select 1, 2, or 3.')

# To launch the application console loop directly from your notebook cell, uncomment and run:
# main_console_app()